In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import pandas as pd
import zipfile
from pathlib import Path
import os
import numpy as np
from ultralytics.utils.metrics import bbox_iou

Paths

In [ ]:
BASE = Path("/content/drive/MyDrive/Final/train_val_split")

TRAIN_ZIP = BASE / "train_512.zip"
VAL_ZIP   = BASE / "val_512.zip"

TRAIN_CSV = BASE / "bboxes_train.csv"
VAL_CSV   = BASE / "bboxes_val.csv"

# YOLO directory
YOLO_DIR = Path("/content/yolo_dataset")
IMG_TRAIN = YOLO_DIR / "images/train"
IMG_VAL = YOLO_DIR / "images/val"
LBL_TRAIN = YOLO_DIR / "labels/train"
LBL_VAL = YOLO_DIR / "labels/val"

for p in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    p.mkdir(parents=True, exist_ok=True)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Extract Images from Zip Files

In [ ]:
def extract_zip(zip_path, out_dir):
    print(f"Extracting {zip_path} → {out_dir}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)

extract_zip(TRAIN_ZIP, IMG_TRAIN)
extract_zip(VAL_ZIP, IMG_VAL)

Extracting /content/drive/MyDrive/Final/train_val_split/train_512.zip → /content/yolo_dataset/images/train
Extracting /content/drive/MyDrive/Final/train_val_split/val_512.zip → /content/yolo_dataset/images/val


Convert Bounding Boxes into YOLO format

In [ ]:
def convert_to_yolo(row, img_w=512, img_h=512):
    x_min, y_min, x_max, y_max = row[['x_min','y_min','x_max','y_max']]
    x_center = (x_min + x_max)/2 / img_w
    y_center = (y_min + y_max)/2 / img_h
    width = (x_max - x_min)/img_w
    height = (y_max - y_min)/img_h
    class_id = 0  # Pneumothorax
    return f"{class_id} {x_center} {y_center} {width} {height}"

def write_labels(csv_path, label_dir):
    df = pd.read_csv(csv_path)
    for pid, group in df.groupby("patientId"):
        label_file = label_dir / f"{pid}.txt"
        with open(label_file, "w") as f:
            for _, row in group.iterrows():
                f.write(convert_to_yolo(row) + "\n")

write_labels(TRAIN_CSV, LBL_TRAIN)
write_labels(VAL_CSV, LBL_VAL)


Writing YOLO labels...


Make yaml file for training

In [ ]:
yaml_path = YOLO_DIR / "data.yaml"
yaml_path.write_text(f"""
train: {IMG_TRAIN}
val: {IMG_VAL}

nc: 1
names: ['pneumonia']
""")

print("Created data.yaml")

Created data.yaml


Train YOLOv8

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data=str(yaml_path),
    imgsz=512,
    epochs=50,
    batch=32,
    device=0,
)

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ff1ad36aa80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
!tar -cvzf runs.tar.gz runs/

runs/
runs/detect/
runs/detect/train/
runs/detect/train/val_batch0_pred.jpg
runs/detect/train/BoxPR_curve.png
runs/detect/train/confusion_matrix.png
runs/detect/train/weights/
runs/detect/train/weights/last.pt
runs/detect/train/weights/best.pt
runs/detect/train/args.yaml
runs/detect/train/val_batch1_pred.jpg
runs/detect/train/BoxP_curve.png
runs/detect/train/train_batch2.jpg
runs/detect/train/BoxR_curve.png
runs/detect/train/train_batch32041.jpg
runs/detect/train/BoxF1_curve.png
runs/detect/train/val_batch0_labels.jpg
runs/detect/train/val_batch2_labels.jpg
runs/detect/train/results.csv
runs/detect/train/train_batch32042.jpg
runs/detect/train/confusion_matrix_normalized.png
runs/detect/train/results.png
runs/detect/train/train_batch1.jpg
runs/detect/train/labels.jpg
runs/detect/train/train_batch0.jpg
runs/detect/train/val_batch1_labels.jpg
runs/detect/train/train_batch32040.jpg
runs/detect/train/val_batch2_pred.jpg


In [ ]:
model_path = "/content/runs/detect/train/weights/best.pt"
dataset_yaml = "/content/yolo_dataset/data.yaml"

model = YOLO(model_path)

# Parse YAML to get validation img path
import yaml
with open(dataset_yaml, "r") as f:
    data_cfg = yaml.safe_load(f)

val_img_dir = Path(data_cfg["val"])
val_lbl_dir = Path(str(val_img_dir).replace("images", "labels"))

# IoU range for evaluation
iou_thresholds = np.arange(0.40, 0.76, 0.05)

# Collect all predictions + GT
all_pred = []
all_gt = []

In [ ]:
# Run predictions on val set

results = model.predict(
    source=str(val_img_dir),
    save=False,
    conf=0.001,
    verbose=False
)

# Loop through each validation image

for r in results:
    img_name = os.path.basename(r.path).replace(".png", ".txt")
    lbl_path = val_lbl_dir / img_name

    if lbl_path.exists():
        gt = np.loadtxt(lbl_path, ndmin=2)
        gx = gt[:, 1] * r.orig_shape[1]
        gy = gt[:, 2] * r.orig_shape[0]
        gw = gt[:, 3] * r.orig_shape[1]
        gh = gt[:, 4] * r.orig_shape[0]
        gt_xyxy = np.column_stack([
            gx - gw / 2, gy - gh / 2,
            gx + gw / 2, gy + gh / 2,
            gt[:, 0]
        ])
        all_gt.append(gt_xyxy)
    else:
        all_gt.append(np.zeros((0, 5)))

    if r.boxes is not None:
        pred = r.boxes.data.cpu().numpy()
        all_pred.append(pred)
    else:
        all_pred.append(np.zeros((0, 6)))

In [ ]:
def compute_ap(gt_boxes, pred_boxes, iou_thres):
    if len(pred_boxes) == 0:
        return 0.0

    pred_boxes = pred_boxes[pred_boxes[:, 4].argsort()[::-1]]

    tp = np.zeros(len(pred_boxes))
    fp = np.zeros(len(pred_boxes))
    matched = []

    for i, p in enumerate(pred_boxes):
        ious = []
        for j, g in enumerate(gt_boxes):
            if g[4] == p[5]:
                iou = bbox_iou(
                    torch.tensor(p[:4]),
                    torch.tensor(g[:4]),
                    xywh=False
                ).item()
                ious.append((iou, j))

        ious = sorted(ious, key=lambda x: x[0], reverse=True)

        if len(ious) > 0 and ious[0][0] >= iou_thres and ious[0][1] not in matched:
            tp[i] = 1
            matched.append(ious[0][1])
        else:
            fp[i] = 1

    # Precision-recall
    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)
    recall = tp_cum / (len(gt_boxes) + 1e-9)
    precision = tp_cum / (tp_cum + fp_cum + 1e-9)

    # AP = area under PR curve
    return np.trapz(precision, recall)

# AP for each IoU threshold
aps = []
for iou in iou_thresholds:
    ap_values = []
    for gt, pred in zip(all_gt, all_pred):
        ap_values.append(compute_ap(gt, pred, iou))
    aps.append(np.mean(ap_values))

map_40_75 = np.mean(aps)

In [ ]:
print(f"mAP@40–75: {map_40_75:.4f}")
print("IoU thresholds:", iou_thresholds)
print("Per-IoU AP:", [round(a, 4) for a in aps])


WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



KeyboardInterrupt: 